# Demographic–Behavioural Integration

## Overview

This notebook integrates the constrained representative demographic profile space with the behavioural evidence derived from NTS0611a, NTS0502b, and NTS0409a.

For a selected feasible demographic profile, trip-purpose probabilities are first retrieved using age and gender. Demographic trip-purpose constraints are then applied according to industry applicability and distance travelled to work. Purpose-conditioned temporal distributions are retained from NTS0502b, while travel-mode distributions are adjusted using age and household car availability.

The resulting output is a constrained demographic–behavioural representation containing:

- eight demographic attributes;
- a constrained trip-purpose probability distribution;
- a temporal distribution for each remaining trip purpose; and
- a constrained travel-mode distribution for each remaining trip purpose.

The framework retains complete probability distributions and does not sample a single trip purpose, start time, or travel mode. Profiles from the constrained representative demographic profile space are used as feasible framework inputs and do not represent synthetic individuals.

This notebook covers demographic–behavioural integration only. Behaviour-to-POI mapping, spatial opportunity adjustment, destination resolution, and final framework integration are handled in subsequent stages.

## 1. Import Libraries

In [1]:
import json
import sys

from pathlib import Path

import pandas as pd


pd.set_option(
    "display.max_columns",
    None,
)

## 2. Set Up File Paths

In [2]:
import sys

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root))

population_file = (
    project_root
    / "output"
    / "demographic_profiles"
    / "constrained_representative_population.parquet"
)

behavioural_evidence_file = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence.json"
)

from src.constraints import (
    apply_trip_purpose_constraint,
    apply_travel_mode_constraint,
)

print("Project root:")
print(project_root)

print("\nPopulation input:")
print(population_file)

print("\nBehavioural evidence input:")
print(behavioural_evidence_file)

print("\nConstraint functions imported successfully.")

Project root:
c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework

Population input:
c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework\output\demographic_profiles\constrained_representative_population.parquet

Behavioural evidence input:
c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework\output\behavioural_evidence\nts_behavioural_evidence.json

Constraint functions imported successfully.


## 3. Load Input Data

In [3]:
population = pd.read_parquet(
    population_file
)

with open(
    behavioural_evidence_file,
    "r",
    encoding="utf-8",
) as file:
    behavioural_evidence = json.load(file)

print(
    "Population loaded:",
    population.shape,
)

print(
    "Trip purposes loaded:",
    len(behavioural_evidence),
)

Population loaded: (2386560, 14)
Trip purposes loaded: 14


## 4. Inspect Input Data

The input datasets are inspected before integration to confirm that the constrained representative profile space contains the eight required demographic attributes and that the behavioural evidence contains the fourteen unified trip purposes and their three probability components.

In [4]:
profile_columns = [
    "Age",
    "Gender",
    "Household Type",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership",
]

missing_columns = [
    column
    for column in profile_columns
    if column not in population.columns
]

if missing_columns:
    raise ValueError(
        "Required demographic columns "
        f"are missing: {missing_columns}"
    )

if len(behavioural_evidence) != 14:
    raise ValueError(
        "Expected 14 unified trip purposes, "
        f"but found {len(behavioural_evidence)}."
    )

display(
    population[
        profile_columns
    ].head()
)

print("Demographic attributes:")
for column in profile_columns:
    print(f"- {column}")

print("\nUnified trip purposes:")
for purpose in behavioural_evidence:
    print(f"- {purpose}")

example_purpose = next(
    iter(behavioural_evidence)
)

print(
    "\nExample purpose:",
    example_purpose,
)

print("Behavioural components:")
for component in (
    behavioural_evidence[
        example_purpose
    ]
):
    print(f"- {component}")

,Age,Gender,Household Type,Economic Activity Status,Industry,Ethnic Group,Distance travelled to work,Car Ownership
0,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,1 car or van in household
1,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,2 cars or vans in household
2,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,3 or more cars or vans in household
3,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,No cars or vans in household
4,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Black, Black British, Black Welsh, Caribbean o...",Does not apply,1 car or van in household


Demographic attributes:
- Age
- Gender
- Household Type
- Economic Activity Status
- Industry
- Ethnic Group
- Distance travelled to work
- Car Ownership

Unified trip purposes:
- Commuting
- Business
- Education
- Escort Education
- Shopping
- Other Escort
- Personal Business
- Visiting Friends at Private Home
- Visiting Friends Elsewhere
- Entertainment or Public Activity
- Sport: Participate
- Holiday: Base
- Day Trip
- Just Walk

Example purpose: Commuting
Behavioural components:
- baseline_profile
- travel_mode_profile
- temporal_pattern_profile


## 5. Define Integration Function

The integration function accepts one feasible representative demographic profile and constructs its constrained behavioural representation.

Trip-purpose probabilities are retrieved from NTS0611a using age and gender before demographic trip-purpose constraints are applied. For every remaining trip purpose, the corresponding temporal distribution is retrieved from NTS0502b and the travel-mode distribution is retrieved from NTS0409a. Travel-mode constraints are then applied using age and household car availability.

The function retains the complete conditional probability distributions and does not sample individual behavioural outcomes.

In [5]:
def construct_constrained_behaviour_profile(
    profile,
    behavioural_evidence,
):
    # Convert the profile to a dictionary.
    if isinstance(profile, pd.Series):
        profile = profile.to_dict()
    else:
        profile = dict(profile)

    # Validate required demographic attributes.
    missing_attributes = [
        attribute
        for attribute in profile_columns
        if attribute not in profile
    ]

    if missing_attributes:
        raise ValueError(
            "Profile is missing required "
            f"attributes: {missing_attributes}"
        )

    # ==========================================
    # 1. Retrieve Trip-Purpose Probabilities
    # ==========================================

    original_purpose_probabilities = {}

    for purpose, evidence in (
        behavioural_evidence.items()
    ):
        matching_records = [
            record
            for record
            in evidence["baseline_profile"]
            if (
                record["Age"]
                == profile["Age"]
            )
            and (
                record["Gender"]
                == profile["Gender"]
            )
        ]

        if len(matching_records) != 1:
            raise ValueError(
                "Expected one baseline record "
                f"for purpose={purpose}, "
                f"age={profile['Age']}, "
                f"gender={profile['Gender']}; "
                f"found {len(matching_records)}."
            )

        original_purpose_probabilities[
            purpose
        ] = float(
            matching_records[0][
                "Trip Purpose Probability"
            ]
        )

    original_purpose_sum = sum(
        original_purpose_probabilities.values()
    )

    if abs(
        original_purpose_sum - 1
    ) >= 1e-8:
        raise ValueError(
            "Original trip-purpose "
            "probabilities do not sum to one. "
            f"Observed sum: {original_purpose_sum}"
        )

    # ==========================================
    # 2. Apply Trip-Purpose Constraints
    # ==========================================

    constrained_purpose_probabilities = (
        apply_trip_purpose_constraint(
            profile=profile,
            trip_purpose_probabilities=(
                original_purpose_probabilities
            ),
        )
    )

    constrained_purpose_sum = sum(
        constrained_purpose_probabilities.values()
    )

    if abs(
        constrained_purpose_sum - 1
    ) >= 1e-8:
        raise ValueError(
            "Constrained trip-purpose "
            "probabilities do not sum to one. "
            f"Observed sum: "
            f"{constrained_purpose_sum}"
        )

    # ==========================================
    # 3. Construct Purpose-Conditioned Profiles
    # ==========================================

    purpose_conditioned_profiles = {}

    metadata_fields = {
        "Year",
        "Purpose",
        "Trip purpose",
    }

    for purpose, purpose_probability in (
        constrained_purpose_probabilities.items()
    ):
        evidence = behavioural_evidence[
            purpose
        ]

        temporal_records = evidence[
            "temporal_pattern_profile"
        ]

        travel_mode_records = evidence[
            "travel_mode_profile"
        ]

        if len(temporal_records) != 1:
            raise ValueError(
                "Expected one temporal record "
                f"for purpose={purpose}; "
                f"found {len(temporal_records)}."
            )

        if len(travel_mode_records) != 1:
            raise ValueError(
                "Expected one travel-mode record "
                f"for purpose={purpose}; "
                f"found {len(travel_mode_records)}."
            )

        temporal_record = temporal_records[0]

        travel_mode_record = (
            travel_mode_records[0]
        )

        temporal_distribution = {
            interval: float(probability)
            for interval, probability
            in temporal_record.items()
            if interval not in metadata_fields
        }

        original_mode_distribution = {
            mode: float(probability)
            for mode, probability
            in travel_mode_record.items()
            if mode not in metadata_fields
        }

        constrained_mode_distribution = (
            apply_travel_mode_constraint(
                age=profile["Age"],
                car_ownership=(
                    profile["Car Ownership"]
                ),
                travel_mode_probabilities=(
                    original_mode_distribution
                ),
            )
        )

        temporal_sum = sum(
            temporal_distribution.values()
        )

        mode_sum = sum(
            constrained_mode_distribution.values()
        )

        if abs(
            temporal_sum - 1
        ) >= 1e-8:
            raise ValueError(
                "Temporal probabilities do not "
                f"sum to one for {purpose}. "
                f"Observed sum: {temporal_sum}"
            )

        if abs(
            mode_sum - 1
        ) >= 1e-8:
            raise ValueError(
                "Travel-mode probabilities do "
                f"not sum to one for {purpose}. "
                f"Observed sum: {mode_sum}"
            )

        purpose_conditioned_profiles[
            purpose
        ] = {
            "trip_purpose_probability": (
                purpose_probability
            ),
            "temporal_distribution": (
                temporal_distribution
            ),
            "travel_mode_distribution": (
                constrained_mode_distribution
            ),
        }

    # ==========================================
    # 4. Return Integrated Representation
    # ==========================================

    return {
        "demographic_profile": {
            attribute: profile[attribute]
            for attribute in profile_columns
        },
        "behavioural_representation": {
            "trip_purpose_distribution": (
                constrained_purpose_probabilities
            ),
            "purpose_conditioned_profiles": (
                purpose_conditioned_profiles
            ),
        },
    }

## 6. Select a Validation Profile

A feasible profile aged 5–11 from a household without a car or van is selected from the constrained representative demographic profile space. This case activates both the trip-purpose and travel-mode constraints and therefore provides a useful initial integration test.

The profile is selected from the existing profile space rather than manually constructed, ensuring that all eight demographic attributes form a logically feasible combination.

In [6]:
validation_candidates = population[
    (
        population["Age"]
        == "5-11"
    )
    & (
        population["Gender"]
        == "Female"
    )
    & (
        population["Car Ownership"]
        == "No cars or vans in household"
    )
]

if validation_candidates.empty:
    raise ValueError(
        "No matching validation "
        "profile was found."
    )

validation_profile = (
    validation_candidates
    .iloc[0][profile_columns]
)

display(
    validation_profile
    .to_frame(
        name="Profile Value"
    )
)

,Profile Value
Age,5-11
Gender,Female
Household Type,Couple household: With dependent children
Economic Activity Status,Does not apply
Industry,Does not apply
Ethnic Group,Mixed or Multiple ethnic groups: Other Mixed o...
Distance travelled to work,Does not apply
Car Ownership,No cars or vans in household


## 7. Construct Constrained Behavioural Representation

The selected demographic profile is passed to the integration function. The resulting representation is inspected to confirm that the relevant trip-purpose and travel-mode constraints have been applied and that all remaining probability distributions are valid.

In [7]:
validation_representation = (
    construct_constrained_behaviour_profile(
        profile=validation_profile,
        behavioural_evidence=(
            behavioural_evidence
        ),
    )
)

trip_purpose_distribution = (
    validation_representation[
        "behavioural_representation"
    ][
        "trip_purpose_distribution"
    ]
)

purpose_conditioned_profiles = (
    validation_representation[
        "behavioural_representation"
    ][
        "purpose_conditioned_profiles"
    ]
)

assert (
    "Commuting"
    not in trip_purpose_distribution
)

assert (
    "Business"
    not in trip_purpose_distribution
)

for purpose, purpose_profile in (
    purpose_conditioned_profiles.items()
):
    mode_distribution = (
        purpose_profile[
            "travel_mode_distribution"
        ]
    )

    assert (
        "Car or van driver"
        not in mode_distribution
    )

    assert (
        "Car or van passenger"
        not in mode_distribution
    )

    assert (
        "Motorcycle"
        not in mode_distribution
    )

    assert (
        "Taxi or minicab"
        in mode_distribution
    )

    assert abs(
        sum(
            purpose_profile[
                "temporal_distribution"
            ].values()
        )
        - 1
    ) < 1e-8

    assert abs(
        sum(mode_distribution.values())
        - 1
    ) < 1e-8

purpose_summary = pd.DataFrame([
    {
        "Trip Purpose": purpose,
        "Trip Purpose Probability": (
            profile[
                "trip_purpose_probability"
            ]
        ),
        "Temporal Interval Count": len(
            profile[
                "temporal_distribution"
            ]
        ),
        "Travel Mode Count": len(
            profile[
                "travel_mode_distribution"
            ]
        ),
        "Trip Purpose Sum": sum(
            trip_purpose_distribution.values()
        ),
        "Temporal Sum": sum(
            profile[
                "temporal_distribution"
            ].values()
        ),
        "Travel Mode Sum": sum(
            profile[
                "travel_mode_distribution"
            ].values()
        ),
    }
    for purpose, profile
    in purpose_conditioned_profiles.items()
])

display(purpose_summary)

print(
    "Original trip-purpose count:",
    len(behavioural_evidence),
)

print(
    "Constrained trip-purpose count:",
    len(trip_purpose_distribution),
)

print(
    "Removed trip purposes:",
    sorted(
        set(behavioural_evidence)
        - set(trip_purpose_distribution)
    ),
)

print(
    "\nDemographic–behavioural "
    "integration: PASS"
)

,Trip Purpose,Trip Purpose Probability,Temporal Interval Count,Travel Mode Count,Trip Purpose Sum,Temporal Sum,Travel Mode Sum
0,Education,0.328955,24,8,1.0,1.0,1.0
1,Escort Education,0.044314,24,8,1.0,1.0,1.0
2,Shopping,0.032020,24,8,1.0,1.0,1.0
3,Other Escort,0.142932,24,8,1.0,1.0,1.0
4,Personal Business,0.075901,24,8,1.0,1.0,1.0
5,Visiting Friends at Private Home,0.106983,24,8,1.0,1.0,1.0
6,Visiting Friends Elsewhere,0.043814,24,8,1.0,1.0,1.0
7,Entertainment or Public Activity,0.078524,24,8,1.0,1.0,1.0
8,Sport: Participate,0.036136,24,8,1.0,1.0,1.0
9,Holiday: Base,0.015216,24,8,1.0,1.0,1.0


Original trip-purpose count: 14
Constrained trip-purpose count: 12
Removed trip purposes: ['Business', 'Commuting']

Demographic–behavioural integration: PASS


## 8. Select Multi-Profile Validation Cases

Four feasible demographic profiles are selected to cover the principal combinations of trip-purpose and travel-mode constraint conditions:

- a young profile without household car availability;
- an adult who works mainly from home and has household car availability;
- a workplace-based adult without household car availability; and
- a workplace-based adult with household car availability as the control case.

All profiles are selected directly from the constrained representative demographic profile space.

In [8]:
NO_CAR_LABEL = (
    "No cars or vans in household"
)

HAS_CAR_LABEL = (
    "1 car or van in household"
)

WFH_LABEL = (
    "Works mainly from home"
)

ADULT_AGE = "23-34"


def select_validation_profile(
    condition,
    profile_type,
):
    candidates = population[
        condition
    ]

    if candidates.empty:
        raise ValueError(
            "No feasible profile found for "
            f"{profile_type}."
        )

    return (
        candidates
        .iloc[0][profile_columns]
    )


young_no_car_profile = (
    select_validation_profile(
        condition=(
            (
                population["Age"]
                == "5-11"
            )
            & (
                population["Gender"]
                == "Female"
            )
            & (
                population["Car Ownership"]
                == NO_CAR_LABEL
            )
        ),
        profile_type=(
            "Young profile without "
            "household car"
        ),
    )
)


wfh_with_car_profile = (
    select_validation_profile(
        condition=(
            (
                population["Age"]
                == ADULT_AGE
            )
            & (
                population["Industry"]
                != "Does not apply"
            )
            & (
                population[
                    "Distance travelled to work"
                ]
                == WFH_LABEL
            )
            & (
                population["Car Ownership"]
                == HAS_CAR_LABEL
            )
        ),
        profile_type=(
            "Adult working mainly from "
            "home with household car"
        ),
    )
)


workplace_no_car_profile = (
    select_validation_profile(
        condition=(
            (
                population["Age"]
                == ADULT_AGE
            )
            & (
                population["Industry"]
                != "Does not apply"
            )
            & (
                ~population[
                    "Distance travelled to work"
                ].isin({
                    "Does not apply",
                    WFH_LABEL,
                })
            )
            & (
                population["Car Ownership"]
                == NO_CAR_LABEL
            )
        ),
        profile_type=(
            "Workplace-based adult "
            "without household car"
        ),
    )
)


workplace_with_car_profile = (
    select_validation_profile(
        condition=(
            (
                population["Age"]
                == ADULT_AGE
            )
            & (
                population["Industry"]
                != "Does not apply"
            )
            & (
                ~population[
                    "Distance travelled to work"
                ].isin({
                    "Does not apply",
                    WFH_LABEL,
                })
            )
            & (
                population["Car Ownership"]
                == HAS_CAR_LABEL
            )
        ),
        profile_type=(
            "Workplace-based adult "
            "with household car"
        ),
    )
)


validation_cases = {
    "Young, no household car": {
        "profile": young_no_car_profile,
        "expected_removed_purposes": {
            "Commuting",
            "Business",
        },
        "expected_removed_modes": {
            "Car or van driver",
            "Car or van passenger",
            "Motorcycle",
        },
    },

    "Adult, work from home, with car": {
        "profile": wfh_with_car_profile,
        "expected_removed_purposes": {
            "Commuting",
        },
        "expected_removed_modes": set(),
    },

    "Adult, workplace, no household car": {
        "profile": workplace_no_car_profile,
        "expected_removed_purposes": set(),
        "expected_removed_modes": {
            "Car or van driver",
        },
    },

    "Adult, workplace, with car": {
        "profile": workplace_with_car_profile,
        "expected_removed_purposes": set(),
        "expected_removed_modes": set(),
    },
}


validation_profiles_table = pd.DataFrame([
    {
        "Profile Type": profile_type,
        "Age": case["profile"]["Age"],
        "Gender": case["profile"]["Gender"],
        "Industry": (
            case["profile"]["Industry"]
        ),
        "Distance travelled to work": (
            case["profile"][
                "Distance travelled to work"
            ]
        ),
        "Car Ownership": (
            case["profile"][
                "Car Ownership"
            ]
        ),
        "Expected Removed Purposes": (
            ", ".join(sorted(
                case[
                    "expected_removed_purposes"
                ]
            ))
            if case[
                "expected_removed_purposes"
            ]
            else "None"
        ),
        "Expected Removed Modes": (
            ", ".join(sorted(
                case[
                    "expected_removed_modes"
                ]
            ))
            if case[
                "expected_removed_modes"
            ]
            else "None"
        ),
    }
    for profile_type, case
    in validation_cases.items()
])

display(validation_profiles_table)

,Profile Type,Age,Gender,Industry,Distance travelled to work,Car Ownership,Expected Removed Purposes,Expected Removed Modes
0,"Young, no household car",5-11,Female,Does not apply,Does not apply,No cars or vans in household,"Business, Commuting","Car or van driver, Car or van passenger, Motor..."
1,"Adult, work from home, with car",23-34,Female,J Information and communication,Works mainly from home,1 car or van in household,Commuting,None
2,"Adult, workplace, no household car",23-34,Female,J Information and communication,10km to less than 30km,No cars or vans in household,None,Car or van driver
3,"Adult, workplace, with car",23-34,Female,J Information and communication,10km to less than 30km,1 car or van in household,None,None


## 9. Validate Multi-Profile Integration

Each validation profile is passed through the complete demographic–behavioural integration function. Validation confirms that the expected trip purposes and travel modes are removed, all adjusted probability distributions sum to one, every remaining trip purpose retains temporal and travel-mode evidence, and unaffected distributions remain unchanged.

In [9]:
integration_validation_results = []

metadata_fields = {
    "Year",
    "Purpose",
    "Trip purpose",
}


for profile_type, case in (
    validation_cases.items()
):
    profile = case["profile"]

    expected_removed_purposes = set(
        case[
            "expected_removed_purposes"
        ]
    )

    expected_removed_modes = set(
        case[
            "expected_removed_modes"
        ]
    )

    representation = (
        construct_constrained_behaviour_profile(
            profile=profile,
            behavioural_evidence=(
                behavioural_evidence
            ),
        )
    )

    behavioural_representation = (
        representation[
            "behavioural_representation"
        ]
    )

    constrained_purposes = (
        behavioural_representation[
            "trip_purpose_distribution"
        ]
    )

    purpose_profiles = (
        behavioural_representation[
            "purpose_conditioned_profiles"
        ]
    )

    # ------------------------------------------
    # Validate trip-purpose constraints
    # ------------------------------------------

    original_purpose_probabilities = {}

    for purpose, evidence in (
        behavioural_evidence.items()
    ):
        matching_records = [
            record
            for record
            in evidence["baseline_profile"]
            if (
                record["Age"]
                == profile["Age"]
            )
            and (
                record["Gender"]
                == profile["Gender"]
            )
        ]

        original_purpose_probabilities[
            purpose
        ] = float(
            matching_records[0][
                "Trip Purpose Probability"
            ]
        )

    actual_removed_purposes = (
        set(original_purpose_probabilities)
        - set(constrained_purposes)
    )

    assert (
        actual_removed_purposes
        == expected_removed_purposes
    )

    assert abs(
        sum(constrained_purposes.values())
        - 1
    ) < 1e-8

    if not expected_removed_purposes:
        assert (
            constrained_purposes
            == original_purpose_probabilities
        )

    # ------------------------------------------
    # Validate temporal and mode distributions
    # ------------------------------------------

    temporal_counts = []
    mode_counts = []

    for purpose, purpose_profile in (
        purpose_profiles.items()
    ):
        temporal_distribution = (
            purpose_profile[
                "temporal_distribution"
            ]
        )

        constrained_mode_distribution = (
            purpose_profile[
                "travel_mode_distribution"
            ]
        )

        original_mode_record = (
            behavioural_evidence[
                purpose
            ][
                "travel_mode_profile"
            ][0]
        )

        original_mode_distribution = {
            mode: float(probability)
            for mode, probability
            in original_mode_record.items()
            if mode not in metadata_fields
        }

        actual_removed_modes = (
            set(original_mode_distribution)
            - set(
                constrained_mode_distribution
            )
        )

        assert (
            actual_removed_modes
            == expected_removed_modes
        )

        assert abs(
            sum(
                temporal_distribution.values()
            )
            - 1
        ) < 1e-8

        assert abs(
            sum(
                constrained_mode_distribution.values()
            )
            - 1
        ) < 1e-8

        if not expected_removed_modes:
            assert (
                constrained_mode_distribution
                == original_mode_distribution
            )

        temporal_counts.append(
            len(temporal_distribution)
        )

        mode_counts.append(
            len(
                constrained_mode_distribution
            )
        )

    assert len(set(temporal_counts)) == 1
    assert len(set(mode_counts)) == 1

    integration_validation_results.append({
        "Profile Type": profile_type,
        "Remaining Purposes": len(
            constrained_purposes
        ),
        "Removed Purposes": (
            ", ".join(sorted(
                actual_removed_purposes
            ))
            if actual_removed_purposes
            else "None"
        ),
        "Temporal Intervals": (
            temporal_counts[0]
        ),
        "Remaining Modes": (
            mode_counts[0]
        ),
        "Removed Modes": (
            ", ".join(sorted(
                expected_removed_modes
            ))
            if expected_removed_modes
            else "None"
        ),
        "Probability Validation": "PASS",
        "Unchanged Validation": (
            "PASS"
            if (
                not expected_removed_purposes
                or not expected_removed_modes
            )
            else "Unchanged Component Validation"
        ),
        "Integration Validation": "PASS",
    })


integration_validation_results = (
    pd.DataFrame(
        integration_validation_results
    )
)

display(
    integration_validation_results
)

if (
    integration_validation_results[
        "Integration Validation"
    ]
    == "PASS"
).all():
    print(
        "Multi-profile demographic–"
        "behavioural integration: PASS"
    )
else:
    print(
        "Multi-profile demographic–"
        "behavioural integration: FAIL"
    )

,Profile Type,Remaining Purposes,Removed Purposes,Temporal Intervals,Remaining Modes,Removed Modes,Probability Validation,Unchanged Validation,Integration Validation
0,"Young, no household car",12,"Business, Commuting",24,8,"Car or van driver, Car or van passenger, Motor...",PASS,Unchanged Component Validation,PASS
1,"Adult, work from home, with car",13,Commuting,24,11,None,PASS,PASS,PASS
2,"Adult, workplace, no household car",14,None,24,10,Car or van driver,PASS,PASS,PASS
3,"Adult, workplace, with car",14,None,24,11,None,PASS,PASS,PASS


Multi-profile demographic–behavioural integration: PASS


## 10. Inspect an Integrated Behavioural Representation

The integrated representation for the young profile without household car availability is inspected in greater detail. Education is used as an example to demonstrate how one remaining trip purpose is represented through its trip-purpose probability, temporal distribution, and constrained travel-mode distribution.

This inspection illustrates the structure passed to subsequent behaviour-to-POI integration without sampling a single behavioural outcome.

In [10]:
example_purpose = "Education"

example_behaviour = (
    validation_representation[
        "behavioural_representation"
    ]
)

example_purpose_distribution = (
    example_behaviour[
        "trip_purpose_distribution"
    ]
)

example_purpose_profile = (
    example_behaviour[
        "purpose_conditioned_profiles"
    ][example_purpose]
)


print("Demographic Profile")

display(
    pd.DataFrame(
        validation_representation[
            "demographic_profile"
        ].items(),
        columns=[
            "Attribute",
            "Profile Value",
        ],
    )
)


print("\nTrip-Purpose Distribution")

trip_purpose_table = pd.DataFrame([
    {
        "Trip Purpose": purpose,
        "Probability": probability,
    }
    for purpose, probability
    in example_purpose_distribution.items()
]).sort_values(
    "Probability",
    ascending=False,
)

display(
    trip_purpose_table.reset_index(
        drop=True
    )
)


print(
    f"\nTemporal Distribution: "
    f"{example_purpose}"
)

temporal_table = pd.DataFrame([
    {
        "Time Interval": interval,
        "Probability": probability,
    }
    for interval, probability
    in example_purpose_profile[
        "temporal_distribution"
    ].items()
])

display(temporal_table)


print(
    f"\nTravel-Mode Distribution: "
    f"{example_purpose}"
)

travel_mode_table = pd.DataFrame([
    {
        "Travel Mode": mode,
        "Probability": probability,
    }
    for mode, probability
    in example_purpose_profile[
        "travel_mode_distribution"
    ].items()
]).sort_values(
    "Probability",
    ascending=False,
)

display(
    travel_mode_table.reset_index(
        drop=True
    )
)


print(
    "\nTrip-purpose sum:",
    sum(
        example_purpose_distribution.values()
    ),
)

print(
    "Temporal sum:",
    sum(
        example_purpose_profile[
            "temporal_distribution"
        ].values()
    ),
)

print(
    "Travel-mode sum:",
    sum(
        example_purpose_profile[
            "travel_mode_distribution"
        ].values()
    ),
)

Demographic Profile


,Attribute,Profile Value
0,Age,5-11
1,Gender,Female
2,Household Type,Couple household: With dependent children
3,Economic Activity Status,Does not apply
4,Industry,Does not apply
5,Ethnic Group,Mixed or Multiple ethnic groups: Other Mixed o...
6,Distance travelled to work,Does not apply
7,Car Ownership,No cars or vans in household



Trip-Purpose Distribution


,Trip Purpose,Probability
0,Education,0.328955
1,Other Escort,0.142932
2,Visiting Friends at Private Home,0.106983
3,Entertainment or Public Activity,0.078524
4,Personal Business,0.075901
5,Just Walk,0.053895
6,Escort Education,0.044314
7,Visiting Friends Elsewhere,0.043814
8,Day Trip,0.041310
9,Sport: Participate,0.036136



Temporal Distribution: Education


,Time Interval,Probability
0,0000 to 0059 (%),0.000000
1,0100 to 0159 (%),0.000000
2,0200 to 0259 (%),0.000000
3,0300 to 0359 (%),0.000306
4,0400 to 0459 (%),0.000044
5,0500 to 0559 (%),0.000112
6,0600 to 0659 (%),0.002540
7,0700 to 0759 (%),0.097998
8,0800 to 0859 (%),0.377559
9,0900 to 0959 (%),0.017586



Travel-Mode Distribution: Education


,Travel Mode,Probability
0,Walk,0.757551
1,Other local bus,0.125771
2,Other Private Transport,0.048151
3,Pedal Cycle,0.021939
4,Surface Rail,0.021709
5,Other Public Transport,0.013316
6,Taxi or minicab,0.011564
7,Non-local bus,0.000000



Trip-purpose sum: 1.0
Temporal sum: 1.0
Travel-mode sum: 1.0


## Summary

This notebook successfully integrated the constrained representative demographic profile space with the behavioural evidence derived from NTS0611a, NTS0502b, and NTS0409a.

For each feasible demographic profile, trip-purpose probabilities were retrieved using age and gender before demographic constraints were applied. Purpose-conditioned temporal distributions were then retained for every remaining trip purpose, while travel-mode distributions were adjusted using age and household car availability.

The integration was validated across four demographic cases. The young profile without household car availability retained 12 trip purposes and 8 travel modes. The adult work-from-home profile retained 13 trip purposes and all 11 travel modes. The workplace-based adult without household car availability retained all 14 trip purposes and 10 travel modes. The workplace-based control profile with household car availability retained all 14 trip purposes and all 11 travel modes.

All trip-purpose, temporal, and travel-mode probability distributions summed to one following constraint application. Alternatives unaffected by the relevant constraints retained their original probability distributions.

The resulting demographic–behavioural representation preserves complete conditional probability distributions and does not sample realised trip purposes, start times, or travel modes. It provides the behavioural input for the subsequent behaviour-to-POI integration stage and does not yet represent a final destination or complete human mobility representation.